## 01_data_ingestion.ipynb

**Purpose:** Load, inspect, validate, and prepare the synthetic force-main dataset for analysis. This notebook is concerned with data integrity, not engineering conclusions.

### Tasks

- [x] Import the required libraries.
- [x] Load `force_main_segments_synthetic.csv`.
- [x] Inspect the first and last records.
- [x] Confirm the expected 240 segments, eight force mains, and 45 fields.
- [x] Examine column names and data types.
- [ ] Identify missing values and calculate missing-value percentages.
- [ ] Distinguish structural missing values in historical repair records from actual data-quality issues.
- [ ] Check duplicate records and segment identifiers.
- [ ] Validate installation years, segment lengths, diameters, pressures, and cost fields.
- [ ] Inspect categorical values, including materials and data-quality flags.
- [ ] Identify inconsistent or suspicious records.
- [ ] Calculate historical years at risk, accounting for installation dates after 2006.
- [ ] Document all data-quality findings.

### Expected outputs

A validated dataset, a summary of data-quality findings, and a data dictionary.

**Boundary:** Descriptive checks such as `head`, `shape`, `info`, missing values, and uniqueness belong here. You are checking whether the dataset is usable, not interpreting failure behavior.

In [2]:
# Import Dependecies
import sys
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

print("Libraries Successfully Loaded!")



Libraries Successfully Loaded!


In [3]:
# Define Data File
RAW_FM_DATA = "../data/raw/force_main_segments_synthetic.csv"

# Load data
df = pd.read_csv(RAW_FM_DATA)

print("Dataframe loaded successfully")

Dataframe loaded successfully


In [5]:
# Inspect first 5 records
df.head(5)

,segment_id,force_main_id,service_area,upstream_lift_station,downstream_facility,installation_year,material,nominal_diameter_in,segment_length_ft,installation_method,...,repair_method,repair_cost_last_usd_2026,estimated_replacement_cost_usd_2026,estimated_planned_rehab_cost_usd_2026,estimated_emergency_repair_cost_usd_2026,estimated_service_disruption_cost_usd_2026,estimated_environmental_response_cost_usd_2026,annual_maintenance_cost_usd_2026,data_quality_flag,notes
0,FM-01-S01,FM-01,North Basin,LS-12,North WRF,1978,DIP,12,1080,Open cut,...,Emergency spool replacement,73000.0,428000,214000,73000,62000,3000,6400,Review,Illustrative synthetic segment; assumptions re...
1,FM-01-S02,FM-01,North Basin,LS-12,North WRF,1980,DIP,12,600,Open cut,...,NaN,NaN,250000,156000,53000,100000,3000,4400,Verified synthetic,Illustrative synthetic segment; assumptions re...
2,FM-01-S03,FM-01,North Basin,LS-12,North WRF,1977,DIP,12,360,Open cut,...,NaN,NaN,259000,106000,53000,47000,2000,5200,Verified synthetic,Illustrative synthetic segment; assumptions re...
3,FM-01-S04,FM-01,North Basin,LS-12,North WRF,1979,DIP,12,560,Open cut,...,NaN,NaN,269000,159000,51000,64000,3000,5500,Verified synthetic,Illustrative synthetic segment; assumptions re...
4,FM-01-S05,FM-01,North Basin,LS-12,North WRF,1981,DIP,12,1100,HDD,...,Cut-out and replace,127000.0,749000,425000,139000,39000,44000,6300,Verified synthetic,Illustrative synthetic segment; assumptions re...


In [6]:
# inspect last 5 records
df.tail(5)

,segment_id,force_main_id,service_area,upstream_lift_station,downstream_facility,installation_year,material,nominal_diameter_in,segment_length_ft,installation_method,...,repair_method,repair_cost_last_usd_2026,estimated_replacement_cost_usd_2026,estimated_planned_rehab_cost_usd_2026,estimated_emergency_repair_cost_usd_2026,estimated_service_disruption_cost_usd_2026,estimated_environmental_response_cost_usd_2026,annual_maintenance_cost_usd_2026,data_quality_flag,notes
235,FM-08-S26,FM-08,Industrial Basin,LS-63,South WRF,1981,DIP,16,640,Open cut,...,NaN,NaN,352000,131000,58000,94000,2000,4400,Verified synthetic,Illustrative synthetic segment; assumptions re...
236,FM-08-S27,FM-08,Industrial Basin,LS-63,South WRF,1982,DIP,16,1020,Open cut,...,Clamp,52000.0,472000,294000,58000,134000,2000,6100,Verified synthetic,Illustrative synthetic segment; assumptions re...
237,FM-08-S28,FM-08,Industrial Basin,LS-63,South WRF,1984,DIP,16,820,Open cut,...,Cut-out and replace,72000.0,305000,128000,58000,124000,3000,5600,Verified synthetic,Illustrative synthetic segment; assumptions re...
238,FM-08-S29,FM-08,Industrial Basin,LS-63,South WRF,1984,DIP,16,260,Open cut,...,NaN,NaN,225000,95000,46000,48000,3000,4600,Verified synthetic,Illustrative synthetic segment; assumptions re...
239,FM-08-S30,FM-08,Industrial Basin,LS-63,South WRF,1985,DIP,16,1000,Open cut,...,Cut-out and replace,59000.0,410000,174000,61000,147000,2000,4700,Verified synthetic,Illustrative synthetic segment; assumptions re...


In [25]:
# Verify the column names
df.columns

Index(['segment_id', 'force_main_id', 'service_area', 'upstream_lift_station',
       'downstream_facility', 'installation_year', 'material',
       'nominal_diameter_in', 'segment_length_ft', 'installation_method',
       'burial_depth_ft', 'operating_pressure_psi', 'surge_pressure_psi',
       'design_pressure_psi', 'average_flow_gpm', 'peak_flow_gpm',
       'velocity_at_peak_fps', 'soil_type', 'soil_corrosivity',
       'groundwater_depth_ft', 'coastal_exposure', 'roadway_class',
       'crossing_type', 'access_difficulty', 'consequence_class',
       'redundant_bypass_available', 'population_served',
       'critical_customer_count', 'condition_score_1_best_5_worst',
       'inspection_year', 'inspection_method',
       'historical_failure_count_2006_2025', 'last_failure_year',
       'last_failure_cause', 'last_repair_year', 'repair_method',
       'repair_cost_last_usd_2026', 'estimated_replacement_cost_usd_2026',
       'estimated_planned_rehab_cost_usd_2026',
       'estimated

In [26]:
# verify information about the dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 45 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   segment_id                                      240 non-null    str    
 1   force_main_id                                   240 non-null    str    
 2   service_area                                    240 non-null    str    
 3   upstream_lift_station                           240 non-null    str    
 4   downstream_facility                             240 non-null    str    
 5   installation_year                               240 non-null    int64  
 6   material                                        240 non-null    str    
 7   nominal_diameter_in                             240 non-null    int64  
 8   segment_length_ft                               240 non-null    int64  
 9   installation_method                             240 no

In [24]:
# Verify the number of force main segments
df["force_main_id"].unique()

<StringArray>
['FM-01', 'FM-02', 'FM-03', 'FM-04', 'FM-05', 'FM-06', 'FM-07', 'FM-08']
Length: 8, dtype: str

In [29]:
# Explore data types
df.dtypes

segment_id                                            str
force_main_id                                         str
service_area                                          str
upstream_lift_station                                 str
downstream_facility                                   str
installation_year                                   int64
material                                              str
nominal_diameter_in                                 int64
segment_length_ft                                   int64
installation_method                                   str
burial_depth_ft                                   float64
operating_pressure_psi                            float64
surge_pressure_psi                                float64
design_pressure_psi                                 int64
average_flow_gpm                                    int64
peak_flow_gpm                                       int64
velocity_at_peak_fps                              float64
soil_type     